# A→B architecture S3/S4: does a learned Perceiver + particle model identify the transform in context?

## Description

<!-- Written before the first run (2026-10-29); kept up to date. Design: docs/design_2026-10-28_A_to_B_architecture.md §2, §8. Issue #18. Follows notebook 08 (S0 PASS, S1 GO). -->

**Question.** S1 showed that the information is there: an exhaustive Bayes learner over shifts, using only the frozen
PFN's likelihood, closes 88–95% of the A-only → oracle gap from 5 A points. Now the transform is a full monotone warp
(S3), then a warp plus an output map (S4), and enumeration no longer scales. Can a *learned* model infer it in context,
in one forward pass, from A and B?
1. Does it register at all: does it close a substantial part of the gap, compared with a non-amortised
   importance-sampling Bayes reference?
2. Can the Perceiver + A-encoder initialisation alone identify the transform (R = 0), or are the evidence blocks needed?
3. How should A be moved into B: is one particle enough (K = 1 vs 8)? Does the LUPI teacher help (with vs without), and
   do latent LUPI targets help on top (`jepa`)?
5. When B is false (unrelated), does the model move to A as A's evidence accumulates (the null particle)?
4. Does accounting for A's evidence in B's context (re-encoding $B \cup A_{\mathrm{in}B}$ per particle) improve the
   predictive?

**Setup.** Frozen `pfn_large` (contexts $n \le 100$); $f \sim$ BNN prior; $N_B = 80$ uniform; A's design uniform;
$n_A \in \{0, 1, 2, 5, 10\}$ (nested, evaluation) / $U\{0..10\}$ (training); 32 queries.
- *S3:* $T$ = per-dimension Kumaraswamy CDF, a full-box monotone bijection, $\log\alpha, \log\beta \sim N(0, 0.5^2)$
  (no sub-windows); $h = \mathrm{id}$.
- *S4:* plus $h$ on $y$: `affine` $h(u) = c + e u$, $e \sim U(0.5, 1)$; `kum` a Kumaraswamy CDF, $\log a, \log b \sim N(0, 0.5^2)$.
  $h$ is applied to the noisy observation, so oracle densities are exact by change of variables.
- *Strata:* `identity` (sanity: the model must not lose the oracle), `warp`, `unrelated` (negative transfer: the model
  must move to A as evidence against B accumulates). Training mixes them 0.15 / 0.70 / 0.15.

*Model* (`experimental/a2b/model.py`):
- a Perceiver over B's frozen PFN token states and $(x, y)$, with $M = 16$ latents;
- an A-encoder;
- $K$ particles with DETR-style init;
- decoders: $T_k(x) = \sigma(\mathrm{logit}\,x + \Delta(\gamma(x), P_k))$, a residual field that is the identity at init, and,
  in S4, a monotone knot map $h_k$ (identity at init);
- $R$ weight-tied evidence blocks: likelihood, PIT, B's hidden state, a lookup into $Z_B$;
- explicit weights $\log w_k = \pi(P_k) + \sum_a \lambda_{ka}$ (pseudo-likelihood, no tempering);
- a **null particle** ("B is unrelated"): the frozen A-only PFN, weighted by its chain-rule evidence
  $\sum_a \log \mathrm{PFN}(y_a \mid A_{<a}, x_a)$ plus a learned offset;
- a Bayes mixture over the $K$ particles and the null as the output.

*Loss (LUPI only, user 2026-10-29; no supervision of transforms, locations or particles):*
- the mixture NLL at every step $r$ (deep supervision);
- on related pairs, also the NLL of the particles-only mixture. Without it, the null particle dominates at init and the
  particles get no gradient: the "easier to ignore B" trap. The related/unrelated label is used only to route training
  losses, which is privileged information at training time.
- the LUPI teacher (`sup = lupi`): a cross-entropy on a y-grid between the model's mixture and the privileged teacher.
  The teacher is PFN($B \cup A_{\mathrm{in}B}$) at $T(x^*)$ through the true $h$ on related pairs (applied to the particles'
  mixture), and the A-only PFN on unrelated pairs (applied to the full mixture). This is the Rao–Blackwell target: same
  optimum as the NLL, lower variance.

12k steps, batch 32, AdamW 3e-4, cosine. Eval: 128 fixed pairs per stratum.

| variant | change from `full` (K = 8, R = 4, NLL + LUPI teacher, predictive from B only) | answers |
|---|---|---|
| `full` | — | 1 |
| `r0` | R = 0: initialisation only | 2 |
| `k1` | K = 1 | 3 |
| `nosup` | NLL only, no LUPI teacher | 3 |
| `context` | last step predicts with PFN($B \cup (T_k(x_a), h_k^{-1}(y_a))$) at $T_k(x^*)$ | 4 |
| `jepa` | + latent LUPI: $\sum_k \mathrm{sg}(w_k) \lVert h_k - h^* \rVert^2$, the particles' last hidden state at $T_k(x^*)$ against the frozen PFN's at $T(x^*)$ with context $B \cup A_{\mathrm{in}B}$ (user's idea, 2026-10-29) | 3 |

*References* (same frozen PFN):
- `a_only`;
- `oracle`: B at the true $T(x^*)$, through the true $h$;
- `teacher_b`: PFN($B \cup A_{\mathrm{in}B}$) at $T(x^*)$, the ceiling for `context`;
- `is_bayes`: importance sampling over 8192 prior draws of $(T, h)$ with pseudo-likelihood weights; the S1 analogue, with
  its ESS reported.

Sweeps:
- S3, $d = 1$: all 6 variants;
- S3, $d = 2$: `full`, `r0`, `context`;
- S4, $d = 1$: $h \in$ {`affine`, `kum`} × {`full`, `nosup`}.

**What to look for.**
1. NLL vs $n_A$: the model at every step $r$ against the references.
2. Gap closure (a_only → oracle) per variant.
3. Location RMSE of the top-weight particle and the best particle (coverage) vs $n_A$ and $r$.
4. S4: the composite KL to the oracle.
5. Training curves (MLflow).
6. Examples of $T_k$ against the true $T$.

**GO / NO-GO** (*drafted by Claude, 2026-10-29, before the first run*; warp stratum, $n_A = 5$, ratio of means over pairs):
- **S3 registers (GO)** if `full` closes ≥ 50% of the a_only → oracle gap for both $d$ *and* the identity stratum stays
  within 0.1 nats/query of `oracle` at every $n_A$. **NO-GO** if it closes < 20% for either $d$. Otherwise **INCONCLUSIVE**.
- **S4 registers (GO)** if `full` closes ≥ 50% for both `affine` and `kum` ($d = 1$). Same NO-GO and INCONCLUSIVE rules.
- Descriptive, not gated:
  - Perceiver init alone: `r0`'s closure as a share of `full`'s.
  - Refinement: `full` − `r0`, paired.
  - K = 1 vs 8 at $n_A \le 2$.
  - LUPI: `nosup` vs `full`; latent LUPI: `jepa` vs `full`.
  - A's evidence: `context` vs `full` at $n_A = 10$, against `teacher_b`.
  - `full` vs `is_bayes`.
  - Moving to A when B is false: the unrelated stratum's NLL against `a_only`, and the null particle's weight, vs $n_A$.
    S1's `bayes_null` reached within 0.38 nats of `a_only` at $n_A = 1$ and within 0.03 from $n_A = 5$.

**Limits.**
- Pilot scale: one seed, 12k steps. A NO-GO here may mean undertrained, not impossible; the training curves will show
  whether it is still improving.
- `is_bayes` with 8192 prior draws is weak where the posterior is sharp (low ESS), mostly at $d = 2$ and large $n_A$, so
  it is a reference, not a ceiling.
- The null particle's evidence (exact chain rule) and the particles' evidence (pseudo-likelihood, which over-counts)
  are on different footings; the learned offsets and prior head must calibrate them.
- A_train reaches A_test's prediction only through the weights and the null particle, plus the re-encoded context in
  `context`. Local discounting of B (a low-rank, top-k residual on B's cache) is the next step and is not tested here.
- The sigmoid parameterisation keeps $T_k$ in $(0, 1)^d$.
- The LUPI teachers use the true $T$ and $h$; this is allowed in training on the prior only.

*History:* a first launch (2026-10-29 10:19) used a location loss on $T_k$ (S3) and a function-space KL on the particles
(S4) as privileged terms, with no null particle. It was stopped after about 15 minutes, before any result, when the user
reaffirmed "LUPI only" and asked that B be discounted as evidence against it accumulates. Nothing from it is reported.


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/a2b/<sweep_id>"  # set after `just snapshot runs/a2b/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.